In [1]:
### set up the notebook
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import pandas as pd
import geopandas as gpd
import scipy.signal
from scipy.interpolate import griddata
import dask as da

from IPython.core.display import display, HTML
display(HTML("<style>.container { width:90% !important; }</style>"))
np.set_printoptions(linewidth=100) 

plt.rcParams.update({'font.size': 14})

da.config.set(**{'array.slicing.split_large_chunks': True})

In [6]:
### function for ACCESS to SWOT spatial interpolation

def interpolate_access_to_swot(access, swot, points):
    access_list = []
    for i in range(len(access['time'])):
#         print(i)
        # select the ACCESS data at the SWOT times
        access_at_swot = access.isel(time=i)

        swot_lat = swot.latitude.values.flatten()
        swot_lon = swot.longitude.values.flatten()

        # interpolate ACCESS to the SWOT locations
        values = access_at_swot.ZWD.values.flatten()
        tmp_ZWD = griddata(points, values, (swot_lon, swot_lat), method='linear')
        tmp_ZWD = tmp_ZWD.reshape(int(len(swot_lon)/69), 69)

        # create a new dataset with the tmp array as the ZWD variable, and swot.num_lines and swot.num_pixels as dimensions
        access_ZWD = xr.Dataset({'ZWD': (['num_lines', 'num_pixels'], tmp_ZWD), 'latitude': swot.latitude, 'longitude': swot.longitude})

        # append to the list
        access_list.append(access_ZWD)

    return access_list

In [4]:
### read in the data

# load the ACCESS data
bs = xr.open_mfdataset('../data_to_publish/ACCESS_ZWD_bst_20230330_20230710.nc')
alb = xr.open_mfdataset('../data_to_publish/ACCESS_ZWD_alb_20230330_20230710.nc')
crp = xr.open_mfdataset('../data_to_publish/ACCESS_ZWD_crp_20230330_20230710.nc')
dvr = xr.open_mfdataset('../data_to_publish/ACCESS_ZWD_dvr_20230330_20230710.nc')

# drop duplicates
bs = bs.drop_duplicates('time')
alb = alb.drop_duplicates('time')
crp = crp.drop_duplicates('time')
dvr = dvr.drop_duplicates('time')

# load the swot data
swot_bs_6 = xr.open_dataset('/data/hendreya/SWOT/SWOT_data/SWOT_L2_subsets_PGC0/SWOT_L2_LR_SSH_Expert_006_PGC0_02_bst.nc')
swot_bs_19 = xr.open_dataset('/data/hendreya/SWOT/SWOT_data/SWOT_L2_subsets_PGC0/SWOT_L2_LR_SSH_Expert_019_PGC0_02_bst.nc')
swot_alb_8 = xr.open_dataset('/data/hendreya/SWOT/SWOT_data/SWOT_L2_subsets_PGC0/SWOT_L2_LR_SSH_Expert_008_PGC0_02_alb.nc')
swot_alb_21 = xr.open_dataset('/data/hendreya/SWOT/SWOT_data/SWOT_L2_subsets_PGC0/SWOT_L2_LR_SSH_Expert_021_PGC0_02_alb.nc')
swot_dvr = xr.open_dataset('/data/hendreya/SWOT/SWOT_data/SWOT_L2_subsets_PGC0/SWOT_L2_LR_SSH_Expert_019_PGC0_02_dvr.nc')
swot_crp = xr.open_dataset('/data/hendreya/SWOT/SWOT_data/SWOT_L2_subsets_PGC0/SWOT_L2_LR_SSH_Expert_006_PGC0_02_crp.nc')

In [5]:
### select ACCESS data at SWOT times

# create arrays of SWOT times
swot_bs_6_times = swot_bs_6.time.mean(axis=1)
swot_bs_19_times = swot_bs_19.time.mean(axis=1)
swot_alb_8_times = swot_alb_8.time.mean(axis=1)
swot_alb_21_times = swot_alb_21.time.mean(axis=1)
swot_dvr_times = swot_dvr.time.mean(axis=1)
swot_crp_times = swot_crp.time.mean(axis=1)

# remove NaNs
swot_bs_6_times = swot_bs_6_times[~np.isnan(swot_bs_6_times)]
swot_bs_19_times = swot_bs_19_times[~np.isnan(swot_bs_19_times)]
swot_alb_8_times = swot_alb_8_times[~np.isnan(swot_alb_8_times)]
swot_alb_21_times = swot_alb_21_times[~np.isnan(swot_alb_21_times)]
swot_dvr_times = swot_dvr_times[~np.isnan(swot_dvr_times)]
swot_crp_times = swot_crp_times[~np.isnan(swot_crp_times)]

# reindex ACCESS to swot times
bs_6_reindexed = bs.reindex(time=swot_bs_6_times.values, method='nearest', tolerance='1H')
bs_19_reindexed = bs.reindex(time=swot_bs_19_times.values, method='nearest', tolerance='1H')
alb_8_reindexed = alb.reindex(time=swot_alb_8_times.values, method='nearest', tolerance='1H')
alb_21_reindexed = alb.reindex(time=swot_alb_21_times.values, method='nearest', tolerance='1H')
crp_reindexed = crp.reindex(time=swot_crp_times.values, method='nearest', tolerance='1H')
dvr_reindexed = dvr.reindex(time=swot_dvr_times.values, method='nearest', tolerance='1H')

# print number of times from each
print(len(swot_bs_6_times), len(swot_bs_19_times), len(swot_alb_8_times), len(swot_alb_21_times), len(swot_dvr_times), len(swot_crp_times))

95 96 96 98 96 94


In [7]:
### interpolate ACCESS to the SWOT locations

# create a 2d array of lat and lon from ACCESS, by flattening the lat and lon grids
bs_lon = bs.lon.values.flatten()
bs_lat = bs.lat.values.flatten()
bs_lon_grid, bs_lat_grid = np.meshgrid(bs_lon, bs_lat)
bs_lon = bs_lon_grid.flatten()
bs_lat = bs_lat_grid.flatten()
bs_points = np.array([bs_lon, bs_lat]).T

alb_lon = alb.lon.values.flatten()
alb_lat = alb.lat.values.flatten()
alb_lon_grid, alb_lat_grid = np.meshgrid(alb_lon, alb_lat)
alb_lon = alb_lon_grid.flatten()
alb_lat = alb_lat_grid.flatten()
alb_points = np.array([alb_lon, alb_lat]).T

crp_lon = crp.lon.values.flatten()
crp_lat = crp.lat.values.flatten()
crp_lon_grid, crp_lat_grid = np.meshgrid(crp_lon, crp_lat)
crp_lon = crp_lon_grid.flatten()
crp_lat = crp_lat_grid.flatten()
crp_points = np.array([crp_lon, crp_lat]).T

dvr_lon = dvr.lon.values.flatten()
dvr_lat = dvr.lat.values.flatten()
dvr_lon_grid, dvr_lat_grid = np.meshgrid(dvr_lon, dvr_lat)
dvr_lon = dvr_lon_grid.flatten()
dvr_lat = dvr_lat_grid.flatten()
dvr_points = np.array([dvr_lon, dvr_lat]).T

bs_6_list = interpolate_access_to_swot(bs_6_reindexed, swot_bs_6, bs_points)
print('bs 6 done')
bs_19_list = interpolate_access_to_swot(bs_19_reindexed, swot_bs_19, bs_points)
print('bs 19 done')
alb_8_list = interpolate_access_to_swot(alb_8_reindexed, swot_alb_8, alb_points)
print('alb 8 done')
alb_21_list = interpolate_access_to_swot(alb_21_reindexed, swot_alb_21, alb_points)
print('alb 21 done')
crp_list = interpolate_access_to_swot(crp_reindexed, swot_crp, crp_points)
print('crp done')
dvr_list = interpolate_access_to_swot(dvr_reindexed, swot_dvr, dvr_points)
print('dvr done')


bs 6 done
bs 19 done
alb 8 done
alb 21 done
crp done
dvr done


In [8]:
### reformat the datasets and save

bs_6_ZWD = xr.concat(bs_6_list, dim='time')
bs_6_ZWD = bs_6_ZWD.assign_coords(time=bs_6_reindexed.time)
bs_19_ZWD = xr.concat(bs_19_list, dim='time')
bs_19_ZWD = bs_19_ZWD.assign_coords(time=bs_19_reindexed.time)
alb_8_ZWD = xr.concat(alb_8_list, dim='time')
alb_8_ZWD = alb_8_ZWD.assign_coords(time=alb_8_reindexed.time)
alb_21_ZWD = xr.concat(alb_21_list, dim='time')
alb_21_ZWD = alb_21_ZWD.assign_coords(time=alb_21_reindexed.time)
crp_ZWD = xr.concat(crp_list, dim='time')
crp_ZWD = crp_ZWD.assign_coords(time=crp_reindexed.time)
dvr_ZWD = xr.concat(dvr_list, dim='time')
dvr_ZWD = dvr_ZWD.assign_coords(time=dvr_reindexed.time)

# save the dataset
bs_6_ZWD.to_netcdf('../data_to_publish/ACCESS_ZWD_bst_6_at_SWOT.nc')
bs_19_ZWD.to_netcdf('../data_to_publish/ACCESS_ZWD_bst_19_at_SWOT.nc')
alb_8_ZWD.to_netcdf('../data_to_publish/ACCESS_ZWD_alb_8_at_SWOT.nc')
alb_21_ZWD.to_netcdf('../data_to_publish/ACCESS_ZWD_alb_21_at_SWOT.nc')
crp_ZWD.to_netcdf('../data_to_publish/ACCESS_ZWD_crp_at_SWOT.nc')
dvr_ZWD.to_netcdf('../data_to_publish/ACCESS_ZWD_dvr_at_SWOT.nc')

/tmp/ipykernel_2827588/410985916.py:17: SerializationWarning: saving variable time with floating point data as an integer dtype without any _FillValue to use for NaNs
  bs_6_ZWD.to_netcdf('../data_to_publish/ACCESS_ZWD_bst_6_at_SWOT.nc')
/tmp/ipykernel_2827588/410985916.py:18: SerializationWarning: saving variable time with floating point data as an integer dtype without any _FillValue to use for NaNs
  bs_19_ZWD.to_netcdf('../data_to_publish/ACCESS_ZWD_bst_19_at_SWOT.nc')
/tmp/ipykernel_2827588/410985916.py:19: SerializationWarning: saving variable time with floating point data as an integer dtype without any _FillValue to use for NaNs
  alb_8_ZWD.to_netcdf('../data_to_publish/ACCESS_ZWD_alb_8_at_SWOT.nc')
/tmp/ipykernel_2827588/410985916.py:20: SerializationWarning: saving variable time with floating point data as an integer dtype without any _FillValue to use for NaNs
  alb_21_ZWD.to_netcdf('../data_to_publish/ACCESS_ZWD_alb_21_at_SWOT.nc')
/tmp/ipykernel_2827588/410985916.py:21: Se